# Test Forecast Request

End-to-end test: pick a model → fetch source data → send `/predict` → poll → visualise.

All runtime parameters (archive URL, input range, step, etc.) come from the model's
`cache_config.json` bundle — no extra configuration needed here.

**Before running:** make sure Docker Compose is up (`docker compose up -d` in `ml-server/`).

---
## 0 — Configuration

In [87]:
import os, json, time, pathlib
from datetime import datetime, timedelta, timezone
from urllib.parse import urlparse, urlunparse
import requests
import pandas as pd
import plotly.graph_objects as go
import mlflow
from mlflow.tracking import MlflowClient
from IPython.display import display

# ── Endpoints ────────────────────────────────────────────────────────────────
MLFLOW_URI    = os.getenv('MLFLOW_TRACKING_URI', 'http://localhost:5050')
ML_SERVER_URL = os.getenv('ML_SERVER_BASE_URL',  'http://localhost:8030')

# ── Override if needed ───────────────────────────────────────────────────────
MODEL_ID_OVERRIDE = os.getenv('MODEL_ID_OVERRIDE', None)   # e.g. 'root_FP_PROJECT_AKMOLA_regions_..._h24_model'
VERSION_ALIAS     = os.getenv('ML_SERVER_VERSION_ALIAS', 'Production')

POLL_INTERVAL = 2.0
MAX_ATTEMPTS  = 60

mlflow.set_tracking_uri(MLFLOW_URI)
client = MlflowClient(tracking_uri=MLFLOW_URI)

print(f'MLflow URI : {MLFLOW_URI}')
print(f'ML Server  : {ML_SERVER_URL}')
print(f'Alias      : {VERSION_ALIAS}')

MLflow URI : http://localhost:5050
ML Server  : http://localhost:8030
Alias      : Production


---
## 1 — Health Check

In [88]:
def check(label, fn):
    try:
        ok, detail = fn()
        print(f"{'✅' if ok else '❌'}  {label}: {detail}")
        return ok
    except Exception as e:
        print(f'❌  {label}: {type(e).__name__}: {e}')
        return False

ok_server = check('ml-server', lambda: (
    (r := requests.get(f'{ML_SERVER_URL}/ui/runtime-status', timeout=5)).status_code == 200,
    f'HTTP {r.status_code}'
))
ok_mlflow = check('MLflow   ', lambda: (
    (r := requests.get(f'{MLFLOW_URI}/health', timeout=5)).status_code == 200,
    f'HTTP {r.status_code}  {r.text.strip()[:40]}'
))

if not (ok_server and ok_mlflow):
    print('\n⚠  Run: cd ml-server && docker compose up -d')
else:
    print('\nAll services UP — ready.')

✅  ml-server: HTTP 200
✅  MLflow   : HTTP 200  OK

All services UP — ready.


---
## 2 — Pick Model from Registry

In [89]:
registered = client.search_registered_models()

rows = []
for rm in registered:
    aliases_flat = ', '.join(
        f'{alias}=v{ver}'
        for ver, aliases in (rm.aliases or {}).items()
        for alias in aliases
    ) or '—'
    rows.append({
        'model_id':       rm.name,
        'latest_version': rm.latest_versions[-1].version if rm.latest_versions else '—',
        'aliases':        aliases_flat,
        'created':        datetime.fromtimestamp(rm.creation_timestamp / 1000, tz=timezone.utc).strftime('%Y-%m-%d') if rm.creation_timestamp else '—',
    })

df_models = pd.DataFrame(rows)
if df_models.empty:
    print('No registered models found.')
    MODEL_ID = MODEL_ID_OVERRIDE
else:
    display(df_models)
    MODEL_ID = MODEL_ID_OVERRIDE or rows[0]['model_id']

print(f'\n→ MODEL_ID: {MODEL_ID}')
if not MODEL_ID:
    print('⚠  Set MODEL_ID_OVERRIDE in Section 0.')

,model_id,latest_version,aliases,created
0,root_FP_PROJECT_AKMOLA_regions_North_Kazakhsta...,17,"1=vProduction, 7=vProduction",2026-05-18
1,root_FP_PROJECT_AKMOLA_regions_North_Kazakhsta...,2,2=vProduction,2026-05-18



→ MODEL_ID: root_FP_PROJECT_AKMOLA_regions_North_Kazakhstan_load_models_P_watt_h24_model


In [90]:
import tempfile
BUNDLE_CONFIG = None

if MODEL_ID:
    try:
        mv = client.get_model_version_by_alias(MODEL_ID, VERSION_ALIAS)
        print(f"Alias '{VERSION_ALIAS}' → version {mv.version}  (run_id={mv.run_id})")

        with tempfile.TemporaryDirectory() as tmpdir:
            for path in ['bundle/configuration/cache_config.json',
                         'bundle/bundle/configuration/cache_config.json']:
                try:
                    local = mlflow.artifacts.download_artifacts(
                        run_id=mv.run_id, artifact_path=path, dst_path=tmpdir)
                    with open(local) as f:
                        BUNDLE_CONFIG = json.load(f)
                    break
                except Exception:
                    pass

        if BUNDLE_CONFIG:
            print('\ncache_config.json:')
            print(json.dumps(BUNDLE_CONFIG, indent=2, ensure_ascii=False))
        else:
            print('⚠  cache_config.json not found in bundle artifacts.')
    except Exception as e:
        print(f"⚠  Could not resolve alias '{VERSION_ALIAS}' for '{MODEL_ID}': {e}")
        print('   Assign a Production alias in MLflow UI (localhost:5050) first.')

Alias 'Production' → version 17  (run_id=28aa0371def54cad8235c1402958d9ca)



cache_config.json:
{
  "step": 3600,
  "input_range": 360,
  "output_range": 36,
  "model_type": "prophet",
  "model_parameters": {
    "seasonality_mode": "multiplicative",
    "changepoint_prior_scale": 0.05,
    "weekly_seasonality": true,
    "daily_seasonality": true
  },
  "fallback": "none",
  "sources": [
    {
      "url": "http://127.0.0.1:7080/api/v1/read/archives",
      "parameters": [
        "/root/FP/PROJECT/AKMOLA/@regions/SevKaz/Load/P_Load/archives/out_value"
      ],
      "pattern": "historical"
    }
  ]
}


---
## 2.1 — Информация о модели

Метаданные из MLflow: дата обучения, параметры, метрики тренировочного прогона.

In [91]:
MLFLOW_RUN = None

if MODEL_ID:
    try:
        _mv = client.get_model_version_by_alias(MODEL_ID, VERSION_ALIAS)
        MLFLOW_RUN = client.get_run(_mv.run_id)
        run = MLFLOW_RUN

        start_dt = datetime.fromtimestamp(run.info.start_time / 1000, tz=timezone.utc)
        end_dt   = (datetime.fromtimestamp(run.info.end_time / 1000, tz=timezone.utc)
                    if run.info.end_time else None)
        duration = f'{(end_dt - start_dt).total_seconds():.0f}s' if end_dt else '—'

        print(f'Model         : {MODEL_ID}')
        print(f'Version       : {_mv.version}  (alias={VERSION_ALIAS})')
        print(f'Run ID        : {_mv.run_id}')
        print(f'Trained       : {start_dt.strftime("%Y-%m-%d %H:%M UTC")}')
        print(f'Training time : {duration}')
        print(f'Run status    : {run.info.status}')

        params  = dict(run.data.params  or {})
        metrics = dict(run.data.metrics or {})
        tags    = {k: v for k, v in (run.data.tags or {}).items()
                   if not k.startswith('mlflow.')}

        if params:
            print('\nParameters:')
            for k, v in sorted(params.items()):
                print(f'  {k}: {v}')

        if metrics:
            print('\nTraining metrics:')
            for k, v in sorted(metrics.items()):
                try:
                    print(f'  {k}: {round(float(v), 4)}')
                except Exception:
                    print(f'  {k}: {v}')

        if tags:
            print('\nTags:')
            for k, v in sorted(tags.items()):
                print(f'  {k}: {v}')

    except Exception as _e:
        print(f'\u26a0  Could not fetch run metadata: {_e}')

Model         : root_FP_PROJECT_AKMOLA_regions_North_Kazakhstan_load_models_P_watt_h24_model
Version       : 17  (alias=Production)
Run ID        : 28aa0371def54cad8235c1402958d9ca
Trained       : 2026-05-18 09:56 UTC
Training time : 0s
Run status    : FINISHED

Parameters:
  row_count: 251
  train_params.changepoint_prior_scale: 0.05
  train_params.daily_seasonality: True
  train_params.seasonality_mode: multiplicative
  train_params.weekly_seasonality: True

Training metrics:
  mae: 11.2502
  mape: 0.0789
  rmse: 13.6269
  wape: 0.0728

Tags:
  bundle_layout: bundle/model + bundle/configuration/cache_config.json
  dataset_hash: 133218f4a9a147b298a4d48ffa8d2ef3d67c34f31e5cadc49ec2786f8a575f45
  dataset_uri: /Users/rustamkrikbayev/Documents/projects/forecast/local/models/root_FP_PROJECT_AKMOLA_regions_North_Kazakhstan_load_models_P_watt_36h/datasets/train_snapshot.parquet
  end_ts: 2026-05-13 15:00:00+00:00
  feature_schema_version: 1
  flow_type: manual_notebook
  model_id: root_FP_PR

---
## 2.5 — Source Data Visualization

Fetch the historical input series using the same parameters the worker will use,
and visualise it before running the forecast.

In [92]:
def normalize_url(url: str) -> str:
    """Rewrite 127.0.0.1/localhost → host.docker.internal when inside Docker."""
    if not url:
        return url
    parsed = urlparse(url)
    if parsed.hostname not in ('127.0.0.1', 'localhost'):
        return url
    if not pathlib.Path('/.dockerenv').exists():
        return url
    netloc = parsed.netloc.replace(parsed.hostname, 'host.docker.internal')
    return urlunparse(parsed._replace(netloc=netloc))


def extract_source_params(cfg: dict) -> dict | None:
    """Pull url, archives, step, input_range, output_range from cache_config."""
    if not cfg:
        return None
    sources = cfg.get('sources', [])
    url, archives = '', []
    if isinstance(sources, list) and sources:
        src      = sources[0]
        url      = src.get('url', '')
        archives = src.get('parameters', [])
        if not archives:
            req = src.get('request', {})
            archives = req.get('archive', req.get('archives', [])) if isinstance(req, dict) else []
    elif isinstance(sources, dict):
        for src in sources.values():
            if isinstance(src, dict):
                url      = src.get('url', '')
                req      = src.get('request', {})
                archives = req.get('archive', req.get('archives', [])) if isinstance(req, dict) else []
                break
    if not archives:
        return None
    return {
        'url':          normalize_url(url),
        'archives':     archives if isinstance(archives, list) else [archives],
        'step':         cfg.get('step', 3600),
        'input_range':  cfg.get('input_range') or cfg.get('output_range', 24),
        'output_range': cfg.get('output_range', 24),
    }


def to_series_df(pts: list) -> pd.DataFrame:
    """Convert [[ts_ms, value, ...], ...] → DataFrame(ts_ms, value, ts).
    Handles 2-column and 3-column point arrays."""
    df = pd.DataFrame(pts).iloc[:, :2]
    df.columns = ['ts_ms', 'value']
    df['ts'] = pd.to_datetime(df['ts_ms'], unit='ms', utc=True).dt.tz_convert(None)
    return df


HISTORICAL_DATA = None
src_params = extract_source_params(BUNDLE_CONFIG)

if src_params is None:
    print('⚠  Cannot extract source parameters from bundle config — skipping historical fetch.')
else:
    history_steps = src_params['input_range']
    now_utc = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
    to_ts   = int(now_utc.timestamp() * 1000)
    from_ts = int((now_utc - timedelta(seconds=history_steps * src_params['step'])).timestamp() * 1000)

    request_body = {
        'from':    from_ts,
        'to':      to_ts,
        'archive': src_params['archives'],
        'step':    src_params['step'],
    }

    print(f"URL      : {src_params['url']}")
    print(f"Archives : {src_params['archives']}")
    print(f"Window   : {history_steps} steps × {src_params['step']}s  ({history_steps * src_params['step'] / 3600:.0f}h)")
    print(f"From     : {datetime.fromtimestamp(from_ts/1000, tz=timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")
    print(f"To       : {datetime.fromtimestamp(to_ts/1000, tz=timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")

    try:
        resp = requests.post(src_params['url'], json=request_body, timeout=30)
        if resp.status_code == 200:
            HISTORICAL_DATA = resp.json()
            total = sum(len(v) for v in HISTORICAL_DATA.values() if isinstance(v, list))
            print(f'\n✅  {len(HISTORICAL_DATA)} series, {total} points')
        else:
            print(f'\n⚠  HTTP {resp.status_code}: {resp.text[:200]}')
    except Exception as e:
        print(f'\n⚠  Could not reach data source: {e}')
        print('   The worker will still attempt the fetch internally.')

URL      : http://127.0.0.1:7080/api/v1/read/archives
Archives : ['/root/FP/PROJECT/AKMOLA/@regions/SevKaz/Load/P_Load/archives/out_value']
Window   : 360 steps × 3600s  (360h)
From     : 2026-05-11 08:00 UTC
To       : 2026-05-26 08:00 UTC

⚠  Could not reach data source: HTTPConnectionPool(host='127.0.0.1', port=7080): Max retries exceeded with url: /api/v1/read/archives (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=7080): Failed to establish a new connection: [Errno 61] Connection refused"))
   The worker will still attempt the fetch internally.


In [93]:
if HISTORICAL_DATA:
    step_sec   = (BUNDLE_CONFIG or {}).get('step', 3600)
    step_label = f"{step_sec // 3600}h" if step_sec >= 3600 else f"{step_sec}s"
    colors     = ['#2ecc71', '#e74c3c', '#9b59b6', '#f39c12', '#1abc9c']
    fig        = go.Figure()

    for i, (name, pts) in enumerate(
        (n, p) for n, p in HISTORICAL_DATA.items() if isinstance(p, list) and p
    ):
        df_s  = to_series_df(pts)
        valid = df_s['value'].dropna()
        label = name.split('/')[-1] if '/' in name else name
        print(f"  {label}: {len(df_s)} pts  "
              f"min={valid.min():.1f}  max={valid.max():.1f}  "
              f"mean={valid.mean():.1f}  nulls={df_s['value'].isna().sum()}")
        fig.add_trace(go.Scatter(
            x=df_s['ts'], y=df_s['value'],
            mode='lines', name=label,
            line=dict(color=colors[i % len(colors)], width=1.5),
        ))

    fig.update_layout(
        title=f'Historical Input Data  |  {MODEL_ID}  |  step={step_label}',
        xaxis_title='Time (UTC)', yaxis_title='Value',
        hovermode='x unified', template='plotly_white', height=400,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    )
    fig.show()
else:
    print('No historical data — proceeding to forecast.')

No historical data — proceeding to forecast.


---
## 3 — Send Forecast Request

All parameters are read from `cache_config.json` by the worker. `object_ref` is optional.

In [98]:
if not MODEL_ID:
    raise RuntimeError('MODEL_ID is not set — see Section 2.')

url    = f'{ML_SERVER_URL}/predict/{MODEL_ID}'
params = {'version_alias': VERSION_ALIAS}

print(f'GET {url}')
print('Params:', json.dumps(params, indent=2))

resp = requests.get(url, params=params, timeout=30)
data = resp.json() if resp.content else {}

print(f'\nHTTP {resp.status_code}')
print(json.dumps(data, indent=2, ensure_ascii=False))

if resp.status_code != 202:
    raise RuntimeError(f'Expected 202, got {resp.status_code}')

TASK_ID = data.get('task_id')
if not TASK_ID:
    raise RuntimeError(f'No task_id in response: {data}')

print(f'\n✅  task_id = {TASK_ID}')

GET http://localhost:8030/predict/root_FP_PROJECT_AKMOLA_regions_North_Kazakhstan_load_models_P_watt_h24_model
Params: {
  "version_alias": "Production"
}

HTTP 202
{
  "status": 202,
  "task_id": "ead8fedfb4064ba482300e733dfce62b",
  "object_ref": null,
  "state": "start"
}

✅  task_id = ead8fedfb4064ba482300e733dfce62b


---
## 4 — Poll for Result

In [99]:
RESULT       = None
FINAL_STATUS = None
tasks_url    = f'{ML_SERVER_URL}/tasks/{TASK_ID}'

print(f'Polling {tasks_url}  (max {MAX_ATTEMPTS} × {POLL_INTERVAL}s):')
print('-' * 60)

for attempt in range(1, MAX_ATTEMPTS + 1):
    r = requests.get(tasks_url, timeout=30)
    d = r.json() if r.content else {}
    FINAL_STATUS = r.status_code

    if r.status_code == 202:
        print(f'  [{attempt:>3}/{MAX_ATTEMPTS}] state={d.get("state", "?")} — processing…')
        time.sleep(POLL_INTERVAL)
        continue

    print(f'  [{attempt:>3}/{MAX_ATTEMPTS}] HTTP {r.status_code}  state={d.get("state", "?")}')    
    RESULT = d

    if r.status_code == 503:
        print('\n⚠  503 — model bundle not available (check MLflow registry and alias).')
        try:
            snap = requests.get(f'{ML_SERVER_URL}/ui/models', timeout=10).json()
            print(json.dumps(snap, indent=2, ensure_ascii=False))
        except Exception as ex:
            print(f'  (could not fetch /ui/models: {ex})')
    break
else:
    raise TimeoutError(f'No result after {MAX_ATTEMPTS} attempts')

print('-' * 60)
print(f'Final status: HTTP {FINAL_STATUS}')

Polling http://localhost:8030/tasks/ead8fedfb4064ba482300e733dfce62b  (max 60 × 2.0s):
------------------------------------------------------------
  [  1/60] HTTP 200  state=done
------------------------------------------------------------
Final status: HTTP 200


---
## 5 — Visualise: Source Data + Forecast

In [100]:
if RESULT is None or FINAL_STATUS != 200:
    print(f'⚠  No result to visualise (HTTP {FINAL_STATUS}).')
    if RESULT:
        print(json.dumps(RESULT, indent=2, ensure_ascii=False))
else:
    output_raw = RESULT.get('data', {}).get('output', RESULT.get('output', []))

    if not output_raw:
        print('⚠  output is empty.')
        print(json.dumps(RESULT, indent=2, ensure_ascii=False))
    else:
        step_sec   = (BUNDLE_CONFIG or {}).get('step', 3600)
        step_label = f"{step_sec // 3600}h" if step_sec >= 3600 else f"{step_sec}s"

        df_fc  = to_series_df(output_raw)
        fig    = go.Figure()
        colors = ['#2ecc71', '#e74c3c', '#9b59b6', '#f39c12']

        # ── Historical input ──────────────────────────────────────────────────
        if HISTORICAL_DATA:
            for i, (name, pts) in enumerate(
                (n, p) for n, p in HISTORICAL_DATA.items() if isinstance(p, list) and p
            ):
                df_s  = to_series_df(pts)
                label = name.split('/')[-1] if '/' in name else name
                fig.add_trace(go.Scatter(
                    x=df_s['ts'], y=df_s['value'],
                    mode='lines', name=f'History: {label}',
                    line=dict(color=colors[i % len(colors)], width=1.5),
                    opacity=0.7,
                ))

        # ── Forecast ─────────────────────────────────────────────────────────
        fig.add_trace(go.Scatter(
            x=df_fc['ts'], y=df_fc['value'],
            mode='lines+markers', name='Forecast',
            line=dict(color='#3498db', width=2.5),
            marker=dict(size=4),
        ))

        # ── History / forecast boundary ───────────────────────────────────────
        # Keep x as datetime-like and add annotation separately to avoid plotly
        # axis-spanning annotation arithmetic on string x-values.
        if HISTORICAL_DATA:
            boundary_ts = df_fc['ts'].min().to_pydatetime()
            fig.add_vline(
                x=boundary_ts,
                line_dash='dash', line_color='#7f8c8d', line_width=1,
            )
            fig.add_annotation(
                x=boundary_ts, y=1.0, xref='x', yref='paper',
                text='forecast start', showarrow=False,
                xanchor='left', yanchor='bottom',
                font=dict(color='#7f8c8d', size=11),
            )

        fig.update_layout(
            title=f'Source + Forecast  |  {MODEL_ID}  |  alias={VERSION_ALIAS}  |  step={step_label}',
            xaxis_title='Time (UTC)', yaxis_title='Value',
            hovermode='x unified', template='plotly_white', height=500,
            legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        )
        fig.show()

        # ── Forecast-only plot (separate view) ─────────────────────────────────
        fig_fc = go.Figure()
        fig_fc.add_trace(go.Scatter(
            x=df_fc['ts'], y=df_fc['value'],
            mode='lines+markers', name='Forecast',
            line=dict(color='#1f77b4', width=2.5),
            marker=dict(size=5),
        ))
        fig_fc.update_layout(
            title=f'Forecast Only  |  {MODEL_ID}  |  alias={VERSION_ALIAS}  |  step={step_label}',
            xaxis_title='Time (UTC)', yaxis_title='Forecast Value',
            hovermode='x unified', template='plotly_white', height=420,
        )
        fig_fc.show()

        conf = RESULT.get('data', RESULT).get('model_confidence')
        print(f'{len(df_fc)} forecast points  |  '
              f'{df_fc["ts"].min()} → {df_fc["ts"].max()}  |  '
              f'confidence={conf}')


        # ── Input data validation + walk-forward backtest (output_range) ─────────
        import numpy as np

        def _safe_mape(y_true, y_pred, eps=1e-9):
            y_true = np.asarray(y_true, dtype=float)
            y_pred = np.asarray(y_pred, dtype=float)
            denom = np.maximum(np.abs(y_true), eps)
            return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

        def _safe_smape(y_true, y_pred, eps=1e-9):
            y_true = np.asarray(y_true, dtype=float)
            y_pred = np.asarray(y_pred, dtype=float)
            denom = np.maximum(np.abs(y_true) + np.abs(y_pred), eps)
            return float(np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0)

        if HISTORICAL_DATA and len(df_fc) >= 2:
            first_name, first_series = next(((n, pts) for n, pts in HISTORICAL_DATA.items() if isinstance(pts, list) and pts), (None, None))
            if first_series:
                df_hist_bt = to_series_df(first_series).sort_values('ts').dropna(subset=['value']).copy()

                # Input validation
                val_errors = []
                if df_hist_bt.empty:
                    val_errors.append('historical series is empty after dropping NaN values')
                if not df_hist_bt['ts'].is_monotonic_increasing:
                    val_errors.append('timestamps are not monotonic increasing')
                dup_cnt = int(df_hist_bt['ts'].duplicated().sum())
                if dup_cnt > 0:
                    val_errors.append(f'duplicate timestamps detected: {dup_cnt}')

                step_sec_expected = int((BUNDLE_CONFIG or {}).get('step', 3600))
                if len(df_hist_bt) >= 2:
                    diffs = df_hist_bt['ts'].diff().dropna().dt.total_seconds().astype(int)
                    bad_steps = int((diffs != step_sec_expected).sum())
                    if bad_steps > 0:
                        val_errors.append(f'irregular step: {bad_steps} intervals differ from step={step_sec_expected}s')

                v = df_hist_bt['value'].astype(float)
                if not np.isfinite(v).all():
                    val_errors.append('non-finite values (inf/-inf) in historical series')
                if (v < 0).any():
                    val_errors.append('negative load values detected (check source units/quality)')

                if val_errors:
                    print('Input validation: FAIL')
                    for e in val_errors:
                        print(f'  - {e}')
                else:
                    print('Input validation: OK')

                # Walk-forward proxy backtest with shift to past by output_range
                h = int((BUNDLE_CONFIG or {}).get('output_range', len(df_fc)))
                h = max(2, min(h, len(df_hist_bt) // 3, len(df_fc)))
                if len(df_hist_bt) >= (h + 24):
                    train = df_hist_bt.iloc[:-h].copy()
                    fact = df_hist_bt.iloc[-h:].copy()

                    # Proxy forecast from train window (mean of latest window)
                    w = min(24, max(3, len(train) // 10))
                    proxy_level = float(train['value'].iloc[-w:].mean())
                    pred_vals = np.repeat(proxy_level, h)

                    pred_df = fact[['ts']].copy()
                    pred_df['pred'] = pred_vals
                    cmp = pred_df.merge(fact[['ts', 'value']], on='ts', how='inner').rename(columns={'value': 'fact'})

                    if len(cmp) >= 2:
                        y_true = cmp['fact'].to_numpy(dtype=float)
                        y_pred = cmp['pred'].to_numpy(dtype=float)
                        err = y_pred - y_true

                        mae = float(np.mean(np.abs(err)))
                        rmse = float(np.sqrt(np.mean(err ** 2)))
                        mape = _safe_mape(y_true, y_pred)
                        smape = _safe_smape(y_true, y_pred)
                        bias = float(np.mean(err))

                        naive_pred = np.repeat(float(train['value'].iloc[-1]), len(cmp))
                        naive_rmse = float(np.sqrt(np.mean((naive_pred - y_true) ** 2)))
                        skill = 1.0 - (rmse / naive_rmse) if naive_rmse > 0 else None

                        # Simple trend slope (for diagnostics: why a line can look flat)
                        x_train_idx = np.arange(len(train), dtype=float)
                        y_train = train['value'].to_numpy(dtype=float)
                        slope, intercept = np.polyfit(x_train_idx, y_train, 1)

                        # Linear regression baseline with calendar features (hour/day-of-week)
                        def _calendar_features(ts_series):
                            ts = pd.to_datetime(ts_series, utc=True)
                            hour = ts.dt.hour.to_numpy()
                            dow = ts.dt.dayofweek.to_numpy()
                            h_oh = np.eye(24, dtype=float)[hour]
                            d_oh = np.eye(7, dtype=float)[dow]
                            return np.concatenate([np.ones((len(ts), 1)), h_oh, d_oh], axis=1)

                        X_train = _calendar_features(train['ts'])
                        y_train = train['value'].to_numpy(dtype=float)
                        beta, *_ = np.linalg.lstsq(X_train, y_train, rcond=None)

                        X_test = _calendar_features(cmp['ts'])
                        linreg_pred = X_test @ beta
                        linreg_err = linreg_pred - y_true
                        linreg_mae = float(np.mean(np.abs(linreg_err)))
                        linreg_rmse = float(np.sqrt(np.mean(linreg_err ** 2)))
                        linreg_mape = _safe_mape(y_true, linreg_pred)
                        linreg_smape = _safe_smape(y_true, linreg_pred)
                        linreg_bias = float(np.mean(linreg_err))
                        linreg_skill = 1.0 - (linreg_rmse / naive_rmse) if naive_rmse > 0 else None

                        print('Walk-forward backtest (shifted to past by output_range):')
                        print(f'  series={first_name}  horizon={len(cmp)} step(s)')
                        print(f'  proxy: MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%  sMAPE={smape:.2f}%  bias={bias:.4f}')
                        print(f'  naive: RMSE={naive_rmse:.4f}  skill_vs_naive={(skill if skill is not None else float("nan")):.4f}')
                        print(f'  baseline: MAE={linreg_mae:.4f}  RMSE={linreg_rmse:.4f}  MAPE={linreg_mape:.2f}%  sMAPE={linreg_smape:.2f}%  bias={linreg_bias:.4f}  skill_vs_naive={(linreg_skill if linreg_skill is not None else float("nan")):.4f}')
                        print(f'  simple_trend_slope={slope:.6f} per step')

                        if rmse > linreg_rmse:
                            print(f'⚠  Warning: forecast RMSE ({rmse:.4f}) is worse than baseline RMSE ({linreg_rmse:.4f}).')
                        else:
                            print(f'✅ Forecast beats baseline by RMSE: {linreg_rmse - rmse:.4f}')

                        fig_bt = go.Figure()
                        fig_bt.add_trace(go.Scatter(x=cmp['ts'], y=cmp['fact'], mode='lines+markers', name='Fact', line=dict(color='#2c3e50', width=2)))
                        fig_bt.add_trace(go.Scatter(x=cmp['ts'], y=cmp['pred'], mode='lines+markers', name='Backtest forecast (proxy)', line=dict(color='#16a085', width=2)))
                        fig_bt.update_layout(
                            title=f'Backtest: Forecast vs Fact (past window, h={len(cmp)})',
                            xaxis_title='Time (UTC)', yaxis_title='Value',
                            hovermode='x unified', template='plotly_white', height=420,
                        )
                        fig_bt.show()


                        # Combined chart: source data + forecast + linear regression baseline
                        fig_combo = go.Figure()
                        fig_combo.add_trace(go.Scatter(
                            x=train['ts'], y=train['value'], mode='lines',
                            name='Source Data (train)', line=dict(color='#7f8c8d', width=1.6)
                        ))
                        fig_combo.add_trace(go.Scatter(
                            x=cmp['ts'], y=cmp['fact'], mode='lines+markers',
                            name='Fact (past window)', line=dict(color='#2c3e50', width=2)
                        ))
                        fig_combo.add_trace(go.Scatter(
                            x=cmp['ts'], y=cmp['pred'], mode='lines+markers',
                            name='Forecast (proxy)', line=dict(color='#16a085', width=2)
                        ))
                        fig_combo.add_trace(go.Scatter(
                            x=cmp['ts'], y=linreg_pred, mode='lines+markers',
                            name='Baseline (linear regression)', line=dict(color='#d35400', width=3, dash='dot')
                        ))
                        fig_combo.update_layout(
                            title='Source Data + Forecast + Linear regression',
                            xaxis_title='Time (UTC)', yaxis_title='Value',
                            hovermode='x unified', template='plotly_white', height=460,
                        )
                        fig_combo.show()

                        fig_err = go.Figure()
                        fig_err.add_trace(go.Scatter(x=np.arange(1, len(cmp) + 1), y=err, mode='lines+markers', name='Proxy error (pred - fact)'))
                        fig_err.add_trace(go.Scatter(x=np.arange(1, len(cmp) + 1), y=linreg_err, mode='lines+markers', name='Baseline error (pred - fact)'))
                        fig_err.add_trace(go.Scatter(x=np.arange(1, len(cmp) + 1), y=(naive_pred - y_true), mode='lines+markers', name='Naive error (pred - fact)'))
                        fig_err.update_layout(
                            title='Backtest Error by Horizon (past window)',
                            xaxis_title='Horizon step', yaxis_title='Error',
                            template='plotly_white', height=360,
                        )
                        fig_err.show()
                    else:
                        print('Walk-forward backtest skipped: could not align forecast timestamps with fact.')
                else:
                    print('Walk-forward backtest skipped: not enough history for output_range split.')

    trimmed = {k: v for k, v in RESULT.items() if k != 'output'}
    if 'data' in trimmed and isinstance(trimmed['data'], dict):
        trimmed['data'] = {k: v for k, v in trimmed['data'].items() if k != 'output'}
    print('\nResult (output omitted):')
    print(json.dumps(trimmed, indent=2, ensure_ascii=False))


36 forecast points  |  2026-05-24 21:00:00 → 2026-05-26 08:00:00  |  confidence=1.0

Result (output omitted):
{
  "status": 200,
  "data": {
    "message": "",
    "model_confidence": 1.0,
    "input_statistics": {
      "series_count": 1,
      "point_count": 360,
      "valid_point_count": 360,
      "start_timestamp": 1778486400000,
      "end_timestamp": 1779778800000,
      "min": 35.9219,
      "max": 426.343,
      "mean": 156.044,
      "std": 37.2443
    },
    "output_statistics": {
      "point_count": 36,
      "valid_point_count": 36,
      "start_timestamp": 1779656400000,
      "end_timestamp": 1779782400000,
      "min": 148.6673,
      "max": 149.8654,
      "mean": 149.2664,
      "std": 0.3556
    },
    "mlflow": {
      "name": "root_FP_PROJECT_AKMOLA_regions_North_Kazakhstan_load_models_P_watt_h24_model",
      "version": "17"
    }
  },
  "object_ref": null,
  "task_id": "ead8fedfb4064ba482300e733dfce62b",
  "state": "done"
}


---
## 5.1 — Исторический бэктест

Запускает тот же адаптерный код, что и воркер (`init_model` + `predict` из `src/adapters`),
на историческом окне, сдвинутом на `output_range` шагов назад относительно текущего момента.
Факт берётся из того же SCADA-источника — это те шаги, которые тогда были будущим.

> **Примечание.** `ProphetAdapter` без `legacy_model` использует `_predict_prophet_simple`
> (линейная экстраполяция по `np.polyfit`). Это текущее поведение продакшена — обученный
> `prophet_model.json` в бандле не загружается в адаптер. Бэктест воспроизводит именно это.

In [101]:
import sys, os, tempfile as _tmpmod
import numpy as _np_bt

# ── 1. Найти src/ ────────────────────────────────────────────────────────────
def _find_src():
    current = os.path.abspath('.')
    for _ in range(8):
        cand = os.path.join(current, 'src')
        if os.path.isdir(os.path.join(cand, 'adapters')):
            return cand
        current = os.path.dirname(current)
    return None

_src = _find_src()
if _src and _src not in sys.path:
    sys.path.insert(0, _src)
    print(f'src added to sys.path: {_src}')
elif _src:
    print(f'src already in sys.path: {_src}')
else:
    print('\u26a0  src/ not found — backtest unavailable')

# ── 2. Скачать полный бандл (model/ + configuration/) ───────────────────────
_BT_BUNDLE_DIR  = None
_BT_MODEL_PATH  = None
_BT_TMPDIR      = None

if MODEL_ID and BUNDLE_CONFIG and _src:
    try:
        _mv_bt = client.get_model_version_by_alias(MODEL_ID, VERSION_ALIAS)
        _BT_TMPDIR = _tmpmod.mkdtemp(prefix='mlserver_bt_')

        for _root in ('bundle', 'bundle/bundle'):
            try:
                _local = mlflow.artifacts.download_artifacts(
                    run_id=_mv_bt.run_id, artifact_path=_root, dst_path=_BT_TMPDIR)
                _bd = pathlib.Path(_local)
                _md = _bd / 'model'
                if _md.exists():
                    _BT_BUNDLE_DIR = str(_bd)
                    _BT_MODEL_PATH = str(_md)
                    print(f'Bundle : {_bd}')
                    print(f'Model  : {list(_md.iterdir())}')
                    break
            except Exception:
                pass

        if not _BT_BUNDLE_DIR:
            print('\u26a0  Model artifacts not found in bundle — backtest skipped')
    except Exception as _e:
        print(f'\u26a0  Bundle download failed: {_e}')

# ── 3. Запустить бэктест ─────────────────────────────────────────────────────
if _BT_BUNDLE_DIR and src_params and BUNDLE_CONFIG:
    try:
        from adapters import load_model_config
        from adapters.model import init_model as _init_model
        from adapters.inference import predict as _adapter_predict

        _cfg = load_model_config(MODEL_ID, bundle_path=_BT_BUNDLE_DIR, require_bundle=True)
        _step_ms  = _cfg.step * 1000
        _h        = _cfg.output_range   # горизонт прогноза
        _inp      = _cfg.input_range    # история для модели

        # ── as_of = output_range шагов назад от текущего часа ────────────────
        _now  = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
        _as_of = _now - timedelta(seconds=_h * _cfg.step)

        # ── Загрузить: history_from → as_of → as_of+h ────────────────────────
        _hist_from = _as_of - timedelta(seconds=_inp * _cfg.step)
        _fact_to   = _as_of + timedelta(seconds=_h   * _cfg.step)

        _body = {
            'from':    int(_hist_from.timestamp() * 1000),
            'to':      int(_fact_to.timestamp()   * 1000),
            'archive': src_params['archives'],
            'step':    _cfg.step,
        }
        print(f'\nas_of   : {_as_of.strftime("%Y-%m-%d %H:%M UTC")}')
        print(f'Input   : {_inp} steps  ({_inp*_cfg.step/3600:.0f} h)')
        print(f'Horizon : {_h} steps  ({_h*_cfg.step/3600:.0f} h)')

        _resp = requests.post(src_params['url'], json=_body, timeout=30)
        _full = _resp.json() if _resp.status_code == 200 else {}

        if not _full:
            print(f'\u26a0  SCADA returned HTTP {_resp.status_code} — backtest skipped')
        else:
            _first_name = next(iter(_full))
            _pts        = _full[_first_name]
            _df_all     = to_series_df(_pts).sort_values('ts').dropna(subset=['value'])

            _df_inp  = _df_all[_df_all['ts'] <= _as_of].iloc[-_inp:]
            _df_fact = _df_all[_df_all['ts'] >  _as_of].iloc[:_h]

            if len(_df_inp) < max(_inp // 4, 2) or len(_df_fact) < 2:
                print(f'\u26a0  Not enough data: input={len(_df_inp)}, actuals={len(_df_fact)}')
            else:
                _ts_arr  = _df_inp['ts_ms'].to_numpy(dtype=_np_bt.int64)
                _val_arr = _df_inp['value'].to_numpy(dtype=float)

                # ── Инициализация модели (тот же путь что в воркере) ─────────
                _model = _init_model(
                    _BT_MODEL_PATH, _step_ms,
                    _cfg.use_dynamic_normalization,
                    _cfg.fallback,
                    _cfg.model_type,
                )

                # ── Инференс ─────────────────────────────────────────────────
                _preds, _pred_ts, _is_match = _adapter_predict(
                    model=_model,
                    timestamps=[_ts_arr],
                    y=[_val_arr],
                    step=_step_ms,
                    output_range=_h,
                    online=False,
                )

                _df_pred = pd.DataFrame({'ts_ms': _pred_ts, 'predicted': _preds})
                _df_pred['ts'] = (pd.to_datetime(_df_pred['ts_ms'], unit='ms', utc=True)
                                  .dt.tz_convert(None))

                _cmp = (_df_pred
                        .merge(_df_fact[['ts', 'value']].rename(columns={'value': 'actual'}),
                               on='ts', how='inner'))

                if len(_cmp) < 2:
                    print('\u26a0  Timestamp join returned < 2 rows — cannot compute metrics')
                else:
                    _yt = _cmp['actual'].to_numpy(dtype=float)
                    _yp = _cmp['predicted'].to_numpy(dtype=float)
                    _e  = _yp - _yt
                    _mae   = float(_np_bt.mean(_np_bt.abs(_e)))
                    _rmse  = float(_np_bt.sqrt(_np_bt.mean(_e ** 2)))
                    _mape  = _safe_mape(_yt, _yp)
                    _smape = _safe_smape(_yt, _yp)
                    _bias  = float(_np_bt.mean(_e))

                    _naive_pred = _np_bt.repeat(float(_val_arr[-1]), len(_cmp))
                    _naive_rmse = float(_np_bt.sqrt(_np_bt.mean((_naive_pred - _yt) ** 2)))
                    _skill = 1.0 - (_rmse / _naive_rmse) if _naive_rmse > 0 else None

                    print(f'\nBacktest results (adapter={type(_model).__name__}  matching={_is_match}):')
                    print(f'  Points : {len(_cmp)} / {_h}')
                    print(f'  MAE    : {_mae:.3f}')
                    print(f'  RMSE   : {_rmse:.3f}')
                    print(f'  MAPE   : {_mape:.2f}%')
                    print(f'  sMAPE  : {_smape:.2f}%')
                    print(f'  Bias   : {_bias:.3f}')
                    if _skill is not None:
                        _tag = '\u2705' if _skill > 0 else '\u26a0'
                        print(f'  Skill  : {_skill:.4f}  {_tag} (vs naive last-value)')

                    # ── График ────────────────────────────────────────────────
                    _fig_bt = go.Figure()
                    _fig_bt.add_trace(go.Scatter(
                        x=_df_inp['ts'], y=_df_inp['value'],
                        mode='lines', name='History (input)',
                        line=dict(color='#95a5a6', width=1.5), opacity=0.8,
                    ))
                    _fig_bt.add_trace(go.Scatter(
                        x=_cmp['ts'], y=_cmp['actual'],
                        mode='lines+markers', name='Actual',
                        line=dict(color='#2c3e50', width=2),
                        marker=dict(size=5),
                    ))
                    _fig_bt.add_trace(go.Scatter(
                        x=_cmp['ts'], y=_cmp['predicted'],
                        mode='lines+markers', name=f'Model ({type(_model).__name__})',
                        line=dict(color='#e67e22', width=2.5, dash='dot'),
                        marker=dict(size=5),
                    ))
                    _fig_bt.add_vline(
                        x=_as_of.isoformat(), line_dash='dash',
                        line_color='#7f8c8d', line_width=1,
                    )
                    _fig_bt.add_annotation(
                        x=_as_of, y=1.0, xref='x', yref='paper',
                        text='as_of', showarrow=False,
                        xanchor='left', yanchor='bottom',
                        font=dict(color='#7f8c8d', size=11),
                    )
                    _fig_bt.update_layout(
                        title=(f'Backtest: Actual vs Model  |  as_of={_as_of.strftime("%Y-%m-%d %H:%M UTC")}'
                               f'  |  MAE={_mae:.1f}  MAPE={_mape:.1f}%'),
                        xaxis_title='Time (UTC)', yaxis_title='Value',
                        hovermode='x unified', template='plotly_white', height=460,
                        legend=dict(orientation='h', yanchor='bottom', y=1.02,
                                    xanchor='right', x=1),
                    )
                    _fig_bt.show()

    except ImportError as _ie:
        print(f'\u26a0  Could not import adapters ({_ie})')
        print('   Make sure src/ is on PYTHONPATH and all dependencies are installed.')
    except Exception as _ex:
        import traceback
        print(f'\u26a0  Backtest failed: {_ex}')
        traceback.print_exc()

src already in sys.path: /Users/rustamkrikbayev/Documents/projects/forecast/ml-server/src


Bundle : /var/folders/z9/_xmzk2xd65b6z8w_nqdgdbkm0000gn/T/mlserver_bt_jq7w055m/bundle
Model  : [PosixPath('/var/folders/z9/_xmzk2xd65b6z8w_nqdgdbkm0000gn/T/mlserver_bt_jq7w055m/bundle/model/metadata.json'), PosixPath('/var/folders/z9/_xmzk2xd65b6z8w_nqdgdbkm0000gn/T/mlserver_bt_jq7w055m/bundle/model/prophet_model.json')]
⚠  Backtest failed: module 'matplotlib' has no attribute 'get_data_path'


Traceback (most recent call last):
  File "/var/folders/z9/_xmzk2xd65b6z8w_nqdgdbkm0000gn/T/ipykernel_40378/2859913337.py", line 56, in <module>
    from adapters import load_model_config
  File "/Users/rustamkrikbayev/Documents/projects/forecast/ml-server/src/adapters/__init__.py", line 3, in <module>
    from .model import init_model
  File "/Users/rustamkrikbayev/Documents/projects/forecast/ml-server/src/adapters/model.py", line 6, in <module>
    from prophet import Prophet
  File "/Users/rustamkrikbayev/Documents/projects/forecast/.venv/lib/python3.13/site-packages/prophet/__init__.py", line 7, in <module>
    from prophet.forecaster import Prophet
  File "/Users/rustamkrikbayev/Documents/projects/forecast/.venv/lib/python3.13/site-packages/prophet/forecaster.py", line 22, in <module>
    from prophet.plot import (plot, plot_components)
  File "/Users/rustamkrikbayev/Documents/projects/forecast/.venv/lib/python3.13/site-packages/prophet/plot.py", line 22, in <module>
    from matp

---
## 6 — Экспорт прогноза

Сохраняет результат последнего прогона в CSV рядом с тетрадкой.

In [102]:
if RESULT is not None and FINAL_STATUS == 200:
    _output_raw = RESULT.get('data', {}).get('output', RESULT.get('output', []))
    if _output_raw:
        _df_export = to_series_df(_output_raw)[['ts', 'value']].copy()
        _df_export.rename(columns={'ts': 'timestamp', 'value': 'forecast'}, inplace=True)

        _export_dir = pathlib.Path('.') / 'exports'
        _export_dir.mkdir(exist_ok=True)

        _ts_min = to_series_df(_output_raw)['ts'].min().strftime('%Y%m%d_%H%M')
        _ts_max = to_series_df(_output_raw)['ts'].max().strftime('%Y%m%d_%H%M')
        _safe_id = MODEL_ID.replace('/', '_').replace('\\', '_')[:80]
        _fname   = f'{_safe_id}__{_ts_min}__{_ts_max}.csv'
        _fpath   = _export_dir / _fname

        _df_export.to_csv(_fpath, index=False)
        print(f'\u2705  Saved {len(_df_export)} rows → {_fpath}')
        display(_df_export.head())
    else:
        print('\u26a0  No forecast output to export.')
else:
    print(f'\u26a0  No completed result (status={FINAL_STATUS}) — nothing to export.')

✅  Saved 36 rows → exports/root_FP_PROJECT_AKMOLA_regions_North_Kazakhstan_load_models_P_watt_h24_model__20260524_2100__20260526_0800.csv


,timestamp,forecast
0,2026-05-24 21:00:00,149.9
1,2026-05-24 22:00:00,149.8
2,2026-05-24 23:00:00,149.8
3,2026-05-25 00:00:00,149.8
4,2026-05-25 01:00:00,149.7
